# Chapter 1 — What is an Agent?

*Controlled decision systems vs language models.*

## Objective

A language model predicts tokens. An agent does something more dangerous and more useful — it acts.

By the end of this notebook you will have:
- Written a 20-line rule-based agent loop using only the standard library.
- Added a governance shim that rejects an unsafe action.
- Seen the difference between the naive loop (`observe → think → act`) and the governed loop (`observe → validate → plan → authorize → act → verify → log`).

## The naive loop

The simplest agent loop is `observe → think → act → observe`. It works for toy problems and fails the moment actions can be dangerous.

In [ ]:
from dataclasses import dataclass

@dataclass
class GridState:
    x: int
    y: int
    goal: tuple[int, int]
    forbidden: tuple[tuple[int, int], ...] = ()

def rule_based_step(s: GridState) -> str:
    '''Greedy step toward goal. No safety awareness.'''
    if s.x < s.goal[0]: return 'right'
    if s.x > s.goal[0]: return 'left'
    if s.y < s.goal[1]: return 'down'
    if s.y > s.goal[1]: return 'up'
    return 'stop'

def apply_move(s: GridState, action: str) -> GridState:
    dx, dy = {'right': (1, 0), 'left': (-1, 0), 'down': (0, 1), 'up': (0, -1)}.get(action, (0, 0))
    return GridState(s.x + dx, s.y + dy, s.goal, s.forbidden)

Now we put a forbidden cell on the agent's path. The naive loop walks through it.

In [ ]:
state = GridState(0, 0, (3, 3), forbidden=((1, 1), (2, 2)))
trace = [(state.x, state.y)]
for _ in range(20):
    a = rule_based_step(state)
    if a == 'stop':
        break
    state = apply_move(state, a)
    trace.append((state.x, state.y))

hit = [p for p in trace if p in state.forbidden]
print('trace:               ', trace)
print('forbidden cells hit: ', hit)

The agent walked right through the forbidden cells. This is the failure mode the rest of the book is designed to prevent.

## The governed loop

The governed loop wraps the same agent in an `authorize` step. Authorization is separate from the agent's logic — it gates actions.

In [ ]:
def authorize(s: GridState, action: str) -> str:
    proposed = apply_move(s, action)
    if (proposed.x, proposed.y) in s.forbidden:
        return 'deny'
    return 'allow'

def alternate(action: str, s: GridState) -> str:
    if action in ('right', 'left'):
        return 'down' if s.y < s.goal[1] else 'up'
    return 'right' if s.x < s.goal[0] else 'left'

state = GridState(0, 0, (3, 3), forbidden=((1, 1), (2, 2)))
trace = [(state.x, state.y)]
denied = []
for _ in range(40):
    a = rule_based_step(state)
    if a == 'stop':
        break
    if authorize(state, a) == 'deny':
        denied.append(((state.x, state.y), a))
        a = alternate(a, state)
        if authorize(state, a) == 'deny':
            print('no safe move; stopping'); break
    state = apply_move(state, a)
    trace.append((state.x, state.y))

print('governed trace:', trace)
print('denied:        ', denied)

## Taxonomy

| System | Main capability |
| --- | --- |
| Language model | predicts next token |
| Chatbot | responds to user |
| Tool-using model | calls external functions |
| Agent | pursues goals through actions |
| Governed agent | acts under explicit constraints, verification and audit |

This book is about turning the right column into a system. The rest of the chapters build each component of the governed loop carefully.

## Anti-patterns flagged here

- Treating the model's reasoning text as evidence (forward reference to Chapter 3).
- Assuming any model is a drop-in for any other. Model capability is a system parameter and dominates downstream behavior.

In [ ]:
# Self-check: the governed loop must reach the goal without entering forbidden cells.
assert (1, 1) not in trace, 'governed loop should not have entered (1, 1)'
assert (2, 2) not in trace, 'governed loop should not have entered (2, 2)'
assert trace[-1] == (3, 3), 'governed loop should reach the goal'
print('OK')